# Bayesian Hyperparameter Space Optimization and Automated Model Auditing Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner
import numpy as np
import pandas as pd

from src import feature_engineering as fe
from src import optuna_optimization as utils
from src import mlflow_utils as mlf_utils

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Path Configuration & MLflow Backend Binding

In [2]:
RANDOM_STATE = 42
N_SPLITS = 5
EXPERIMENT_NAME = "customer-churn-optuna"

from pathlib import Path

def _find_project_root():
    """Find the project root by looking for pyproject.toml."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (pyproject.toml)")

ROOT_DIR = str(_find_project_root())
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# Set the tracking URI immediately to lock it to SQLite
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Explicitly create the experiment with your designated mlartifacts folder path
experiment_id = mlf_utils.init_mlflow_experiment(
    EXPERIMENT_NAME,
    DB_PATH,
    ARTIFACTS_DIR,
)

# Active the experiment scope
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:///Users/hector.vargas/repos/ml_hands_on_project/mlartifacts', creation_time=1781736737792, experiment_id='1', last_update_time=1781736737792, lifecycle_stage='active', name='customer-churn-optuna', tags={}, trace_location=None, workspace='default'>

## 2. Custom Source Code Ingestion

In [3]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

## 2. Orchestrate and Initialize Search Parameter Studies

In [4]:
selected_features = [
    # Binary
    "is_silver",
    "is_germany",
    "is_spain",
    "Num_Of_Products_1",
    "Num_Of_Products_2",
    "Num_Of_Products_3",
    "Num_Of_Products_4",

    # Continuous
    "Age_x_IsActive",
    "Balance_per_Product",
    "CreditScore_per_Age",
    "Inactive_x_Balance",
    "CreditScore_x_Age",
    "Products_per_Tenure",
]
# Schema Baseline Columns Definitions
nomod_columns = []
dummyfy_columns = ['Gender']
norm_std_columns = ['Point Earned', 'Satisfaction Score', 'EstimatedSalary']

# Initialize the Feature Engineer class with the desired subset strings
feature_engineer_object = fe.DynamicFeatureEngineer(selected_features=selected_features)
binary_features = feature_engineer_object._get_all_binary_features()
continuous_features = feature_engineer_object._get_all_continuous_features()

# FIX: Set current_layout explicitly before instantiating your classes
current_layout = {
    "passthrough": nomod_columns + binary_features,
    "standard_scale": norm_std_columns + continuous_features,
    "one_hot_encode": dummyfy_columns
}

EXPERIMENT_REGISTRY = {
    "experiment_1": current_layout
}

# Initialize Integrated MLflow Tracking Integration Callbacks
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="average_precision",
    create_experiment=False,
    mlflow_kwargs={
        "nested": True,
        "experiment_id": experiment_id,
    },
)

study = optuna.create_study(
    study_name="customer-churn-rf-search-v1",
    direction="maximize",
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=0),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

# FIXED: Removed the invalid FULL_REGISTRY argument variable completely
objective_function = utils.ObjectiveCV(
    X=X_train,
    y=y_train,
    current_layout=current_layout,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_16904/2697242773.py:41: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-06-18 04:05:24,393] A new study created in memory with name: customer-churn-rf-search-v1


## 3. Run Optimization Search Workspace Execution Window

In [6]:
with mlflow.start_run(
    run_name="optuna_search_parent",
    experiment_id=experiment_id,
):    
    study.optimize(
        objective_function,
        n_trials=100,
        callbacks=[mlflow_callback]
    )

    # Log global best performance summary attributes
    mlflow.log_metric("best_auc", study.best_value)
    mlflow.log_params(study.best_params)

    # FIXED: Reconstruct your optimized topology without passing FULL_REGISTRY
    best_pipeline = utils.build_pipeline(
        trial=study.best_trial, 
        current_layout=current_layout,
        random_state=RANDOM_STATE
    )

    # Log downstream outputs using your fixed evaluation function signature
    utils.evaluate_and_log_best_model(
        best_pipeline=best_pipeline,
        X_train=X_train, 
        y_train=y_train,
        X_test=X_test, 
        y_test=y_test,
        current_layout=current_layout,
    )

[I 2026-06-18 04:10:30,076] Trial 1 finished with value: 0.6788379596098221 and parameters: {'scaler': 'minmax', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 293, 'rf_max_depth': 12, 'rf_min_samples_split': 25, 'rf_min_samples_leaf': 1}. Best is trial 1 with value: 0.6788379596098221.
[I 2026-06-18 04:10:31,734] Trial 2 finished with value: 0.6953347996684984 and parameters: {'scaler': 'std', 'encoder': 'no_drop', 'model': 'xgb', 'xgb_n_estimators': 1059, 'xgb_learning_rate': 0.029667626364545993, 'xgb_subsample': 0.9644741157888952, 'xgb_colsample_bytree': 0.47092407909780626, 'xgb_gamma': 2.0842892970704363, 'xgb_reg_alpha': 6.78905327169848e-07, 'xgb_reg_lambda': 1.9069966103000426e-07, 'xgb_scale_pos_weight': 1.992587980696507}. Best is trial 2 with value: 0.6953347996684984.
[I 2026-06-18 04:10:34,082] Trial 3 finished with value: 0.6800919420716183 and parameters: {'scaler': 'robust', 'encoder': 'no_drop', 'model': 'rf', 'rf_n_estimators': 295, 'rf_max_depth': 13, '

## 4. Display Session Optimization Diagnostics Results

In [7]:
print(f"\nTop Optimization Average Precision Score Target: {study.best_value:.4f}")
print("\nOptimal Parameter Combinations Selected:")
for parameter_key, parameter_value in study.best_params.items():
    print(f" * {parameter_key}: {parameter_value}")


Top Optimization Average Precision Score Target: 0.6992

Optimal Parameter Combinations Selected:
 * scaler: minmax
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 1378
 * xgb_learning_rate: 0.029146078705537502
 * xgb_subsample: 0.9506417296176162
 * xgb_colsample_bytree: 0.46961460485789147
 * xgb_gamma: 3.4463899418096147
 * xgb_reg_alpha: 5.24142723644156e-06
 * xgb_reg_lambda: 6.537562546758999e-07
 * xgb_scale_pos_weight: 1.7574589020922013


In [9]:
import pandas as pd
import numpy as np


def suggest_numeric_ranges(
    study,
    top_quantile=0.95,
    boundary_threshold=0.15,
    expansion_factor=0.5,
):
    """
    Suggest new Optuna search ranges for numeric parameters.

    Parameters
    ----------
    study : optuna.study.Study

    top_quantile : float
        Use top X% of trials to infer the optimum region.

    boundary_threshold : float
        Fraction of range considered "close to boundary".

    expansion_factor : float
        How much to expand the range when boundary pressure exists.

    Returns
    -------
    pd.DataFrame
    """

    df = study.trials_dataframe()

    completed = df[df["state"] == "COMPLETE"].copy()

    threshold = completed["value"].quantile(top_quantile)

    top_trials = completed[completed["value"] >= threshold]

    results = []

    for col in completed.columns:

        if not col.startswith("params_"):
            continue

        if not pd.api.types.is_numeric_dtype(completed[col]):
            continue

        all_values = completed[col].dropna()

        if len(all_values) < 5:
            continue

        top_values = top_trials[col].dropna()

        current_min = all_values.min()
        current_max = all_values.max()

        span = current_max - current_min

        if span == 0:
            continue

        median_top = top_values.median()

        relative_position = (
            median_top - current_min
        ) / span

        if relative_position < boundary_threshold:

            action = "move_left"

            suggested_min = current_min - expansion_factor * span
            suggested_max = current_max

        elif relative_position > (1 - boundary_threshold):

            action = "move_right"

            suggested_min = current_min
            suggested_max = current_max + expansion_factor * span

        else:

            q10 = top_values.quantile(0.10)
            q90 = top_values.quantile(0.90)

            concentration = (q90 - q10) / span

            if concentration < 0.40:

                action = "narrow"

                padding = (q90 - q10) * 0.20

                suggested_min = q10 - padding
                suggested_max = q90 + padding

            else:

                action = "keep"

                suggested_min = current_min
                suggested_max = current_max

        results.append(
            {
                "parameter": col.replace("params_", ""),
                "current_min": current_min,
                "current_max": current_max,
                "best_median": median_top,
                "action": action,
                "suggested_min": suggested_min,
                "suggested_max": suggested_max,
            }
        )

    return (
        pd.DataFrame(results)
        .sort_values(
            ["action", "parameter"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
    )

In [10]:
suggestions = suggest_numeric_ranges(study)

display(suggestions)

,parameter,current_min,current_max,best_median,action,suggested_min,suggested_max
0,xgb_scale_pos_weight,1.612775e+00,2.093801,1.884823e+00,keep,1.612775,2.093801
1,xgb_colsample_bytree,4.500792e-01,0.595474,4.693476e-01,move_left,0.377382,0.595474
2,xgb_reg_alpha,1.111976e-08,0.000976,3.821738e-06,move_left,-0.000488,0.000976
3,xgb_reg_lambda,1.096211e-09,0.000096,3.933390e-07,move_left,-0.000048,0.000096
4,xgb_gamma,1.946135e+00,3.497692,3.469111e+00,move_right,1.946135,4.273471
5,xgb_n_estimators,8.180000e+02,1400.000000,1.351000e+03,move_right,818.000000,1691.000000
6,xgb_learning_rate,2.525295e-02,0.038387,2.905535e-02,narrow,0.027132,0.033827
7,xgb_subsample,9.402209e-01,0.979183,9.535300e-01,narrow,0.947944,0.958271


In [11]:
import mlflow
import pandas as pd

experiment = mlflow.get_experiment_by_name(
    "customer-churn-optuna"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.average_precision DESC"]
)

runs.head()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.average_precision,metrics.train_auc,metrics.test_auc,metrics.test_recall,...,tags.xgb_scale_pos_weight_distribution,tags.mlflow.source.name,tags.datetime_start,tags.xgb_reg_alpha_distribution,tags.xgb_colsample_bytree_distribution,tags.xgb_reg_lambda_distribution,tags.rf_min_samples_leaf_distribution,tags.rf_max_depth_distribution,tags.rf_min_samples_split_distribution,tags.rf_n_estimators_distribution
0,06c556a0778d40f8b0cf995cd241d3f1,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 10:11:13.817000+00:00,2026-06-18 10:11:13.831000+00:00,0.699156,NaN,NaN,NaN,...,"FloatDistribution(high=2.1, log=False, low=1.6...",6.0-havg-optuna-model.ipynb,2026-06-18 04:11:13.482855,"FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",None,None,None,None
1,26bdc3ce8a654612ad24b3e3a7f5db56,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 10:11:04.237000+00:00,2026-06-18 10:11:04.252000+00:00,0.698590,NaN,NaN,NaN,...,"FloatDistribution(high=2.1, log=False, low=1.6...",6.0-havg-optuna-model.ipynb,2026-06-18 04:11:03.885858,"FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",None,None,None,None
2,85745016cb6649c4b136f83762e0f84e,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 10:11:14.162000+00:00,2026-06-18 10:11:14.176000+00:00,0.698588,NaN,NaN,NaN,...,"FloatDistribution(high=2.1, log=False, low=1.6...",6.0-havg-optuna-model.ipynb,2026-06-18 04:11:13.833159,"FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",None,None,None,None
3,41ac25e7175f45d3bc1632974a3aded8,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 10:11:09.620000+00:00,2026-06-18 10:11:09.634000+00:00,0.698556,NaN,NaN,NaN,...,"FloatDistribution(high=2.1, log=False, low=1.6...",6.0-havg-optuna-model.ipynb,2026-06-18 04:11:09.253835,"FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",None,None,None,None
4,17247689b9d8481bb2fe461a7bbfebd0,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 10:10:55.616000+00:00,2026-06-18 10:10:55.632000+00:00,0.698493,NaN,NaN,NaN,...,"FloatDistribution(high=2.1, log=False, low=1.6...",6.0-havg-optuna-model.ipynb,2026-06-18 04:10:55.314026,"FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",None,None,None,None


In [12]:
run_id = runs.iloc[0]["run_id"]

run = mlflow.get_run(run_id)

print(run.data.metrics)

{'average_precision': 0.6991562082699793}


In [13]:
metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
runs[metric_cols].head()

,metrics.average_precision,metrics.train_auc,metrics.test_auc,metrics.test_recall,metrics.train_f1,metrics.test_f1,metrics.test_accuracy,metrics.test_precision,metrics.best_auc,metrics.train_accuracy,metrics.train_precision,metrics.train_recall
0,0.699156,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.698590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.698588,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.698556,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.698493,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# from src import feature_engineering as fe
# fe.remove_recent_runs(EXPERIMENT_NAME, 100000)

In [14]:
best_per_model = (
    runs
    .sort_values("metrics.average_precision", ascending=False)
    .groupby("params.model")
    .head(1)
    .loc[:, [
        "params.model",
        "metrics.average_precision",
        "run_id"
    ]]
)

print(best_per_model)

     params.model  metrics.average_precision                            run_id
0             xgb                   0.699156  06c556a0778d40f8b0cf995cd241d3f1
2934           rf                   0.686205  5cb82f8537ea4716ad885624b26d893a


In [15]:
rf_runs = (
    runs[runs["params.model"] == "rf"]
    .sort_values(
        "metrics.average_precision",
        ascending=False
    )
)

rf_runs[
    [
        "run_id",
        "metrics.average_precision",
        "params.rf_max_depth",
        "params.rf_n_estimators",
        "params.rf_min_samples_split",
        "params.rf_min_samples_leaf",
    ]
].head(10)

,run_id,metrics.average_precision,params.rf_max_depth,params.rf_n_estimators,params.rf_min_samples_split,params.rf_min_samples_leaf
2934,5cb82f8537ea4716ad885624b26d893a,0.686205,12,289,21,1
2935,9178c6b545e9488199ea70f7a733306f,0.686170,12,289,21,1
2936,fddd6fc6d4c14697872f6771dc600a09,0.685322,10,290,26,1
2937,f8677fec7bfa43728b4713a14ca13838,0.685260,10,286,26,1
2938,f253ea8b976b4b6daab4a78da835b1c5,0.685060,10,281,26,1
2939,eee44615234f449ba265430decc1451c,0.685035,10,282,22,1
2940,335ff430ff644cbfaa4439707102550c,0.685023,11,288,22,1
2941,d50a2b1d5e4a44e9af90e689dbe3bcc2,0.684835,9,285,28,1
2942,b7682ddf01494bcc9ff4a13641724457,0.684724,13,282,25,2
2943,f9485f74137a44eda09e7a5bb1ded2ac,0.684724,13,282,25,2


In [16]:
xgb_runs = (
    runs[runs["params.model"] == "xgb"]
    .sort_values(
        "metrics.average_precision",
        ascending=False
    )
)

xgb_runs[
    [
        "run_id",
        "metrics.average_precision",
        "params.xgb_n_estimators",
        "params.xgb_learning_rate",
        "params.xgb_gamma",
        "params.xgb_scale_pos_weight",
    ]
].head(10)

,run_id,metrics.average_precision,params.xgb_n_estimators,params.xgb_learning_rate,params.xgb_gamma,params.xgb_scale_pos_weight
0,06c556a0778d40f8b0cf995cd241d3f1,0.699156,1378,0.029146078705537502,3.4463899418096147,1.7574589020922013
1,26bdc3ce8a654612ad24b3e3a7f5db56,0.698590,1338,0.028964611370395472,3.496942962532014,2.006132610767263
2,85745016cb6649c4b136f83762e0f84e,0.698588,1364,0.027713331951805337,3.362024311088895,1.7635141084989447
3,41ac25e7175f45d3bc1632974a3aded8,0.698556,1330,0.03446667719912811,3.4918327712538835,2.0676453846058243
4,17247689b9d8481bb2fe461a7bbfebd0,0.698493,1150,0.026556391586552448,3.2761028269264987,1.7994516532658982
5,988f7ca4e01d4420b90ed3791487052d,0.698478,1156,0.0294425759625736,3.2801037042599455,1.9411776671579646
6,2fe23022da4d43339efeda05636c14b6,0.698449,1342,0.03000701323334874,3.4941483246301144,1.92874964971701
7,c8f3a5d89c6a48f9992ec4e038ac5572,0.698428,1383,0.029152053959253965,3.450409267188936,1.6627440038467483
8,697a276746a24bc681e0b296536dd5c8,0.698421,1331,0.03038513443842619,3.3090581675351722,2.032640537537498
9,7d047fd3dab840afbe250dac78e748ff,0.698287,1308,0.029042371510434303,3.2983842074140384,1.7245362126746375


In [17]:
leaderboard = (
    runs[
        [
            "params.model",
            "metrics.average_precision",
            "run_id",
        ]
    ]
    .sort_values(
        "metrics.average_precision",
        ascending=False
    )
)

leaderboard.head(20)

,params.model,metrics.average_precision,run_id
0,xgb,0.699156,06c556a0778d40f8b0cf995cd241d3f1
1,xgb,0.698590,26bdc3ce8a654612ad24b3e3a7f5db56
2,xgb,0.698588,85745016cb6649c4b136f83762e0f84e
3,xgb,0.698556,41ac25e7175f45d3bc1632974a3aded8
4,xgb,0.698493,17247689b9d8481bb2fe461a7bbfebd0
5,xgb,0.698478,988f7ca4e01d4420b90ed3791487052d
6,xgb,0.698449,2fe23022da4d43339efeda05636c14b6
7,xgb,0.698428,c8f3a5d89c6a48f9992ec4e038ac5572
8,xgb,0.698421,697a276746a24bc681e0b296536dd5c8
9,xgb,0.698287,7d047fd3dab840afbe250dac78e748ff
